# L18 demo: prompting vs RAG, measured

One engineering corpus, one gold set, two systems scored the same way. This is the
decision framework made concrete: when does retrieval earn its keep over a bare prompt?

> Companion notes: [`notes.md`](notes.md). Fine-tuning is the third lever; the notes
> explain when to use it, and why it is not what you reach for here.

No GPU, no network, no API key: retrieval and scoring are computed locally.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


## Setup

In [ ]:
import re, json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
np.random.seed(0)

## The corpus

A handful of engineering-reference snippets, the kind an engineer looks things up in.
Each carries a source tag so a grounded answer can cite it. Some are distractors: near
the queries in wording but not the answer.

In [ ]:
CORPUS = [
  ('PlumbCode 7.3', 'Cold bending of annealed copper tube. The minimum centerline bend '
   'radius for 12 mm outside-diameter annealed copper tube is 45 mm. For 15 mm tube it is 60 mm.'),
  ('PipeSpec 4.1', 'Schedule 40 carbon steel pipe, ASTM A53. The maximum allowable working '
   'pressure for 2-inch Schedule 40 pipe at 200 C is 2.4 MPa; at 20 C it is 3.1 MPa.'),
  ('FastenGuide 2.2', 'Grade 8.8 M12 hex bolts, lightly oiled, shall be tightened to a '
   'torque of 86 N-m. Grade 10.9 M12 bolts to 121 N-m.'),
  ('GasketMan 3.5', 'Spiral-wound gaskets with flexible graphite filler are rated for '
   'continuous service from -200 C to 450 C.'),
  ('PumpManual 9.1', 'The required NPSH for the CP-4L pump at rated flow is 3.2 m. Available '
   'NPSH must exceed this value by a margin of 0.6 m.'),
  # distractors: similar vocabulary, not the answer
  ('PlumbCode 7.1', 'Copper tube shall be cut square and deburred before bending. Annealed '
   'tube bends cold; hard-drawn tube requires a bending spring or fittings.'),
  ('PipeSpec 1.2', 'Carbon steel pipe shall be marked with the heat number, schedule, and '
   'specification. Schedule 40 is the most common wall thickness for general service.'),
  ('FastenGuide 1.1', 'Bolt torque depends on grade, lubrication, and thread pitch. Always '
   'use a calibrated wrench; dry threads need more torque than oiled ones.'),
]
SOURCES = [c[0] for c in CORPUS]
TEXTS = [c[1] for c in CORPUS]
print(f'{len(CORPUS)} chunks ({sum(t.count(".") for t in TEXTS)} sentences)')

## The gold set

The questions an engineer actually asks. Five are **knowledge lookups** whose answers live
in the corpus (or, for one, deliberately do not). One is a **formatting** task that needs
no lookup at all. Each has a hand-written correct answer, so we can score automatically.

In [ ]:
GOLD = [
  ('what is the minimum bend radius for 12 mm copper tube?', '45 mm', 'knowledge'),
  ('maximum allowable working pressure of 2-inch schedule 40 pipe at 200 C?', '2.4 MPa', 'knowledge'),
  ('tightening torque for a grade 8.8 M12 bolt?', '86 N-m', 'knowledge'),
  ('lower temperature limit of a spiral-wound graphite gasket?', '-200 C', 'knowledge'),
  ('required NPSH for the CP-4L pump at rated flow?', '3.2 m', 'knowledge'),
  # deliberately NOT in the corpus: a grounded system must decline
  ('flash point of ISO VG 46 hydraulic oil?', 'not in the provided documents', 'knowledge-absent'),
  # a formatting task: no lookup needed, the base model can already do it
  ('normalize the log "brng making noise on P-101" to component/symptom', 'bearing/noise', 'formatting'),
]
print(f'{len(GOLD)} gold queries')

## System 1: prompting only

The base model answers from what it already knows. It handles the formatting task, but it
has never seen these particular specs, so on the lookups it does what a confident model
does without grounding: it guesses. These canned answers stand in for a real base model
with no access to the corpus.

In [ ]:
# what a base model without the corpus tends to produce: plausible, wrong
PROMPT_ONLY = {
  'what is the minimum bend radius for 12 mm copper tube?': 'about 3 times the diameter, so ~36 mm',
  'maximum allowable working pressure of 2-inch schedule 40 pipe at 200 C?': 'roughly 2.0 MPa',
  'tightening torque for a grade 8.8 M12 bolt?': 'approximately 60 N-m',
  'lower temperature limit of a spiral-wound graphite gasket?': 'around -50 C',
  'required NPSH for the CP-4L pump at rated flow?': 'typically 2 to 3 m',
  'flash point of ISO VG 46 hydraulic oil?': 'about 210 C',   # confident and unverifiable
  'normalize the log "brng making noise on P-101" to component/symptom': 'bearing/noise',
}
def answer_prompting(query):
    return PROMPT_ONLY[query]

## System 2: RAG

Real retrieval, then a grounded read. We TF-IDF the corpus, retrieve the top-k chunks for
the query, then pick the single sentence in those chunks most similar to the query. A
confidence threshold makes the system **decline** when nothing matches well, which is what
keeps it from inventing an answer that is not in the documents. The formatting task has no
answer to retrieve, so RAG falls back to the same normalization the prompt does.

In [ ]:
vec = TfidfVectorizer(stop_words='english').fit(TEXTS)
chunk_mat = vec.transform(TEXTS)

def retrieve(query, k=3):
    sims = cosine_similarity(vec.transform([query]), chunk_mat)[0]
    top = np.argsort(sims)[::-1][:k]
    return [(SOURCES[i], TEXTS[i]) for i in top]

def sentences(chunks):
    out = []
    for src, text in chunks:
        for s in re.split(r'(?<=[.;]) ', text):
            if s.strip():
                out.append((src, s.strip()))
    return out

def answer_rag(query, k=3, threshold=0.12):
    if 'normalize the log' in query:                 # formatting: nothing to ground
        return 'bearing/noise', None
    chunks = retrieve(query, k)
    cand = sentences(chunks)
    svec = vec.transform([s for _, s in cand])
    sims = cosine_similarity(vec.transform([query]), svec)[0]
    best = int(sims.argmax())
    if sims[best] < threshold:
        return 'not in the provided documents', None
    return cand[best][1], cand[best][0]              # grounded sentence + source

## Score them on the same gold set

A prediction counts as correct if the gold answer string appears in it (for the absent
query, the system must decline). Same metric, same data, both systems.

In [ ]:
def correct(pred, gold):
    return gold.lower() in pred.lower()

rows = []
for query, gold, cat in GOLD:
    p = answer_prompting(query)
    r, src = answer_rag(query)
    rows.append((cat, query, gold, p, correct(p, gold), r, src, correct(r, gold)))

print(f'{"category":16} {"prompting":10} {"RAG":6}')
for cat in ['knowledge', 'knowledge-absent', 'formatting']:
    sub = [x for x in rows if x[0] == cat]
    pc = sum(x[4] for x in sub); rc = sum(x[7] for x in sub)
    print(f'{cat:16} {pc}/{len(sub):<8} {rc}/{len(sub)}')

In [ ]:
# the detail, one row per query
for cat, q, gold, p, pok, r, src, rok in rows:
    print(f'Q: {q}')
    print(f'   gold     : {gold}')
    print(f'   prompting: {p[:70]:70} {"OK" if pok else "X"}')
    print(f'   RAG      : {(r[:60] + (" [" + src + "]" if src else ""))[:70]:70} {"OK" if rok else "X"}')
    print()

## Read the result

On the knowledge lookups RAG grounds its answers in a cited chunk while the bare prompt
guesses every one, and on the absent query RAG declines where the prompt confidently
invents a flash point. RAG scores 4 of 5 rather than a clean sweep, and the miss is worth
keeping: the bolt-torque query retrieved a *distractor* sentence ("Bolt torque depends on
grade, lubrication, and thread pitch") instead of the one with the actual 86 N-m value, so
the grounded answer is on-topic but wrong. That is L17's lesson showing up here: retrieval
quality is not free, and a distractor that shares the query's words can outrank the answer.
On the formatting task the two tie, because there is nothing to retrieve and a decent
prompt already does the job. The framework falls out of the table: **knowledge is RAG's
job; behavior and format are the prompt's, and fine-tuning's at scale.**

In [ ]:
import matplotlib.pyplot as plt
cats = ['knowledge', 'knowledge-absent', 'formatting']
labels = ['knowledge\nlookup', 'absent\n(must decline)', 'formatting']
prompting = [sum(x[4] for x in rows if x[0] == c) / max(1, sum(1 for x in rows if x[0] == c)) for c in cats]
rag = [sum(x[7] for x in rows if x[0] == c) / max(1, sum(1 for x in rows if x[0] == c)) for c in cats]
x = np.arange(len(cats)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(x - w/2, [v*100 for v in prompting], w, label='prompting', color='#5c5c5c')
ax.bar(x + w/2, [v*100 for v in rag], w, label='RAG', color='#c41230')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('accuracy (%)'); ax.set_ylim(0, 105)
ax.set_title('Prompting vs RAG on the same gold set'); ax.legend(frameon=False)
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
plt.show()

---

## Takeaway

The lever follows the need. When the task is **knowledge** the model lacks, retrieval wins
and, just as important, lets the system cite its source and decline when it does not know.
When the task is **behavior or format**, a good prompt already suffices, and fine-tuning is
the same lever scaled up for when the format must be baked in across thousands of calls.
The mistake the notes warn about is reaching for fine-tuning to inject knowledge: this
bake-off is the measured version of why that is the wrong tool. Assignment A9 builds the
RAG half of this for real.